# Préparer les références régionales

Ce dossier produit un **livrable de données**, distinct des exemples de Clermont-Ferrand. Notre périmètre est toute l'Auvergne-Rhône-Alpes.

Nous reprenons les stations Citiz fournies dans le dépôt et les contours IGN déjà préparés. Une station sera une ligne du tableau d'indicateurs, identifiée par `station_uid`.

Choisir **Python (IGN / OSM)** et exécuter les cellules dans l'ordre. Les données sont locales : aucun téléchargement n'est réalisé ici.

## 1. Localiser le dépôt

Les fichiers d'entrée restent dans leurs dossiers d'origine. Les fichiers livrables sont écrits dans `regional-data/data/processed/` ; relancer les cellules remplace ces sorties.

In [1]:
from pathlib import Path
import json
import pandas as pd
import geopandas as gpd
from IPython.display import display

current_directory = Path.cwd().resolve()
project_directory = None
for directory in [current_directory, *current_directory.parents]:
    if (directory / "regional-data" / "AGENTS.md").exists():
        project_directory = directory
        break
if project_directory is None:
    raise FileNotFoundError("Open this notebook from the repository directory.")
module_directory = project_directory / "regional-data"
output_directory = module_directory / "data" / "processed"
output_directory.mkdir(parents=True, exist_ok=True)

## 2. Lire et contrôler les stations

Les coordonnées sont en longitude/latitude. Nous signalons les lignes sans coordonnées exploitables, au lieu de les transformer en positions fictives. `station_uid` est la clé conservée pour les jointures avec les tables de l'équipe.

In [2]:
stations_path = project_directory / "output" / "citiz_stations_geo.csv"
stations = pd.read_csv(stations_path, dtype={"station_uid": "string", "station_id": "string"})
assert stations["station_uid"].notna().all()
assert stations["station_uid"].is_unique
stations["lat"] = pd.to_numeric(stations["lat"], errors="coerce")
stations["lon"] = pd.to_numeric(stations["lon"], errors="coerce")
valid_coordinates = stations["lat"].between(-90, 90) & stations["lon"].between(-180, 180)
stations.loc[~valid_coordinates].to_csv(output_directory / "stations_invalid_coordinates.csv", index=False)
valid_stations = stations.loc[valid_coordinates].copy()
station_points = gpd.GeoDataFrame(
    valid_stations,
    geometry=gpd.points_from_xy(valid_stations["lon"], valid_stations["lat"]),
    crs="EPSG:4326",
)
print(f"Input stations: {len(stations)}, invalid coordinates: {(~valid_coordinates).sum()}")

Input stations: 1614, invalid coordinates: 0


## 3. Vérifier le périmètre avec l'IGN

Nous sélectionnons les stations par leurs coordonnées dans le contour IGN, pas seulement par l'étiquette régionale d'un ancien export. `intersects` conserve aussi les points sur la frontière.

Le contour est projeté en Lambert-93 pour calculer une **marge de 1 km**. Les équipements situés juste hors de la région peuvent être proches d'une station régionale : ils doivent rester disponibles pour les comptages.

In [3]:
region = gpd.read_file(project_directory / "ign-osm" / "data" / "processed" / "region_84.geojson")
assert len(region) == 1 and region.geometry.is_valid.all()
region_geometry = region.geometry.iloc[0]
regional_stations = station_points.loc[station_points.geometry.intersects(region_geometry)].copy()
assert regional_stations["station_uid"].is_unique
assert not regional_stations.empty
regional_stations = regional_stations.rename(columns={"lat": "latitude", "lon": "longitude"})
columns = ["station_uid", "source", "station_id", "name", "latitude", "longitude", "address", "capacity", "extracted_at", "geometry"]
regional_stations = regional_stations[columns]

metric_region = region.to_crs(epsg=2154)
counting_area = metric_region.copy()
counting_area["geometry"] = metric_region.geometry.buffer(1000)
region.to_file(output_directory / "aura_region.geojson", driver="GeoJSON")
counting_area.to_crs(epsg=4326).to_file(output_directory / "aura_counting_area.geojson", driver="GeoJSON")
regional_stations.to_file(output_directory / "citiz_stations_aura.geojson", driver="GeoJSON")
regional_stations.drop(columns="geometry").to_csv(output_directory / "citiz_stations_aura.csv", index=False)
print(f"Spatially selected AURA stations: {len(regional_stations)}")
display(regional_stations.drop(columns="geometry").head())

Spatially selected AURA stations: 612


,station_uid,source,station_id,name,latitude,longitude,address,capacity,extracted_at
61,provider_12:1914,provider_12,1914,Gare de Massiac-Blesle,45.252732,3.196774,Pl. des Pupilles de la Nation,1,2026-10-19 14:43:54.580579+00:00
64,provider_5:715,provider_5,715,SNCF Gare du Clapier - Augustin Dupré,45.437710,4.380850,17 Avenue Augustin Dupré,2,2026-10-19 14:43:54.580579+00:00
65,provider_5:716,provider_5,716,Grüner [Station électrique],45.440159,4.399269,"14 Rue de la Montat, 42023 Saint-Étienne",2,2026-10-19 14:43:54.580579+00:00
66,provider_5:717,provider_5,717,Avenue de Cran (20),45.905804,6.118441,"20 avenue de Cran, 74000 Annecy",2,2026-10-19 14:43:54.580579+00:00
67,provider_5:718,provider_5,718,Place des Romains-Sports,45.908855,6.121918,1 avenue du parc des Sports 74000 ANNECY,2,2026-10-19 14:43:54.580579+00:00


## 4. Fournir les communes et vérifier les exports

Les communes sont exportées en **GeoPackage** (`.gpkg`), un fichier géographique compact qui conserve les formes et les attributs. Les codes INSEE restent du texte.

Les stations sont une photographie issue du travail de l'équipe ; ce notebook ne rafraîchit pas leurs flux GBFS.

In [4]:
municipalities = gpd.read_file(project_directory / "ign-osm" / "data" / "processed" / "municipalities_84.geojson")
municipalities.to_file(output_directory / "aura_municipalities.gpkg", layer="municipalities", driver="GPKG")
saved_stations = gpd.read_file(output_directory / "citiz_stations_aura.geojson")
assert set(saved_stations["station_uid"]) == set(regional_stations["station_uid"])
assert len(gpd.read_file(output_directory / "aura_municipalities.gpkg")) == len(municipalities)
summary = {"station_count": len(regional_stations), "municipality_count": len(municipalities), "invalid_coordinate_count": int((~valid_coordinates).sum()), "station_source": "output/citiz_stations_geo.csv", "region_code": "84", "counting_margin_m": 1000}
(output_directory / "reference_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print(summary)

{'station_count': 612, 'municipality_count': 4025, 'invalid_coordinate_count': 0, 'station_source': 'output/citiz_stations_geo.csv', 'region_code': '84', 'counting_margin_m': 1000}


## Suite

Le notebook 02 extraira les objets OSM à l'intérieur de la région **et de sa marge**. Les bâtiments ou lieux seront fournis avec leurs identifiants et leurs géométries pour permettre d'autres calculs sur de futurs emplacements.